# PM2.5 零樣本預測：完整實驗（Colab Enterprise）

1. 先把 `pm25_project.zip` 上傳到左側檔案瀏覽器（與此 notebook 同一層）。
2. 從上到下依序執行。每個實驗儲存格都**可重跑續跑**：已完成的「測站 × 模型」會自動跳過。
3. 最後一格會產生 `pm25_results.zip`，下載後交給 Claude 彙整。
4. **跑完務必關閉執行階段**，否則會持續扣額度。

In [ ]:
# 1. 檢查 GPU 與 Python 版本
!nvidia-smi || echo '沒有 GPU，將使用 CPU（較慢但可跑）'
!python --version

In [ ]:
# 2. 解壓縮專案
import os
!unzip -q -o pm25_project.zip -d pm25
os.chdir('pm25')
!ls

In [ ]:
# 3. 安裝套件（torch 若已預裝會沿用）
!pip install -q -e '.[models]'
!python -c "from pm25.models.device import default_device; print('device =', default_device())"

## 實驗
Chronos 與 XGBoost 必須在**不同行程**執行，每個 `!python` 指令就是獨立行程，所以照下列儲存格分開跑即可。

In [ ]:
# 4. Persistence 基準（幾秒）
!python -m pm25.experiment --family baseline

In [ ]:
# 5. Chronos-Bolt Small/Base、Chronos-2（單變量／共變數／未來風場 oracle）
# GPU 上約 10–30 分鐘；第一次會下載模型權重
!python -m pm25.experiment --family chronos

In [ ]:
# 6. XGBoost（CPU）
!python -m pm25.experiment --family xgboost

In [ ]:
# 7. LSTM（有 GPU 會自動使用）
!python -m pm25.experiment --family lstm

## 彙整與下載

In [ ]:
# 8. 產生指標表（results/tables）
!python -m pm25.evaluation.report

In [ ]:
# 9. 打包結果並下載
!zip -q -r ../pm25_results.zip results
print('已產生 pm25_results.zip（在左側檔案瀏覽器上一層）')
try:
    from google.colab import files
    files.download('../pm25_results.zip')
except Exception as exc:
    print('自動下載失敗，請在左側檔案瀏覽器對 pm25_results.zip 按右鍵下載：', exc)

## 結束
到 Colab Enterprise →「執行階段」把這個執行階段**刪除或停止**，避免繼續計費。